# Notebook 5: Structure Validation — Ramachandran Plot & Z-Score
**ProteinIQ Project**

## Background
Structure validation checks whether a predicted or experimental model is physically plausible:

- **Ramachandran plot** (Ramachandran et al., 1963): plots backbone dihedral angles φ and ψ. In high-quality structures, >90% of residues lie in the 'core allowed' regions.
- **Z-score** (ProSA, Wiederstein & Sippl 2007): measures how much the model's energy deviates from a database of well-folded proteins. Z < −4 → well folded.
- **PROCHECK grading**: A (≥90% core, ≤2% outliers) → F (<60% core).

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', 'backend'))
from models.ramachandran import (compute_phi_psi, classify_residue,
                                  validate_structure, compute_z_score)
from models.energy_minimizer import random_coil_coords
np.random.seed(42)

## 1. Ramachandran Allowed Regions — Reference Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
# Draw allowed regions
regions = [
    {'phi':(-90,-30), 'psi':(-60,-10), 'label':'α-helix', 'color':'#D85A30', 'alpha':0.25},
    {'phi':(-160,-60),'psi':(100,180),  'label':'β-strand', 'color':'#378ADD', 'alpha':0.25},
    {'phi':(30,90),   'psi':(20,80),   'label':'L-helix',  'color':'#1D9E75', 'alpha':0.25},
    {'phi':(-90,-50), 'psi':(130,170), 'label':'PPII',     'color':'#9B59B6', 'alpha':0.15},
]
for r in regions:
    x0,x1 = r['phi']; y0,y1 = r['psi']
    ax.add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,
                 color=r['color'],alpha=r['alpha'],label=r['label'],zorder=1))
ax.axvline(0, color='gray', lw=0.5, alpha=0.4)
ax.axhline(0, color='gray', lw=0.5, alpha=0.4)
ax.set_xlim(-180,180); ax.set_ylim(-180,180)
ax.set_xlabel('φ (phi) — degrees', fontsize=12)
ax.set_ylabel('ψ (psi) — degrees', fontsize=12)
ax.set_title('Ramachandran plot — allowed regions', fontsize=13)
ax.set_xticks(range(-180,181,60))
ax.set_yticks(range(-180,181,60))
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()

## 2. Validate Structures with Different Quality Levels

In [ ]:
# Simulate 3 structures: good (helix), mixed, random
seq = 'ACDEFGHIKLMNPQRSTVWYACDEFGH'

structures = {
    'Good (helical)':  random_coil_coords(len(seq), seed=7),
    'Mixed':           random_coil_coords(len(seq), seed=99),
    'Random coil':     np.random.randn(len(seq), 3) * 5,
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ss_colors = {'H':'#D85A30','E':'#378ADD','C':'#888780','terminal':'gray'}

for ax, (title, coords) in zip(axes, structures.items()):
    report = validate_structure(coords, sequence=seq)
    # Draw region backgrounds
    for r in [{'phi':(-90,-30),'psi':(-60,-10),'c':'#D85A30'},
              {'phi':(-160,-60),'psi':(100,180),'c':'#378ADD'},
              {'phi':(30,90),'psi':(20,80),'c':'#1D9E75'}]:
        x0,x1=r['phi']; y0,y1=r['psi']
        ax.add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,
                     color=r['c'],alpha=0.12,zorder=0))
    # Plot residues
    for r in report.per_residue:
        if r['phi'] and r['psi']:
            reg_color = {'core':'#1D9E75','allowed':'#378ADD',
                         'generous':'#F0A500','outlier':'#D85A30'}
            c = reg_color.get(r['region'],'gray')
            ax.scatter(r['phi'], r['psi'], color=c, s=35, alpha=0.8, edgecolors='none')
            ax.annotate(r['aa'], (r['phi'], r['psi']), fontsize=6, alpha=0.7)
    ax.set_xlim(-180,180); ax.set_ylim(-180,180)
    ax.axvline(0,color='gray',lw=0.5,alpha=0.3); ax.axhline(0,color='gray',lw=0.5,alpha=0.3)
    ax.set_xlabel('φ'); ax.set_ylabel('ψ')
    ax.set_title(f'{title}\nGrade {report.grade}  Core: {report.pct_core:.0f}%  '
                  f'Z: {report.z_score}  Outliers: {report.pct_outliers:.0f}%')
    ax.grid(alpha=0.1)

plt.suptitle('Ramachandran plot comparison across structure qualities', fontsize=12)
plt.tight_layout()
plt.savefig('../data/figures/ramachandran_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Z-Score Distribution — What Makes a Good Score?

In [ ]:
# Generate Z-scores for structures of different lengths and qualities
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

lengths = [10, 20, 30, 50, 80, 100]
for trial in range(5):
    zscores = [compute_z_score(random_coil_coords(n, seed=trial))['z_score'] for n in lengths]
    axes[0].plot(lengths, zscores, 'o-', alpha=0.6, lw=1.5)
axes[0].axhline(-3, color='#1D9E75', ls='--', lw=2, label='Well-folded threshold (−3)')
axes[0].axhline(0, color='#D85A30', ls='--', lw=1.5, label='Poor quality (0)')
axes[0].set_xlabel('Sequence length'); axes[0].set_ylabel('Z-score')
axes[0].set_title('Z-score vs chain length (random coil)')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.2)

# Grade distribution
labels = ['Grade A\n(≥90% core)', 'Grade B\n(80–90%)', 'Grade C\n(60–80%)', 'Grade F\n(<60%)']
values = [35, 28, 22, 15]   # example distribution from PDB survey
colors = ['#1D9E75','#378ADD','#F0A500','#D85A30']
bars = axes[1].bar(labels, values, color=colors, alpha=0.8, edgecolor='none')
for bar, v in zip(bars, values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v}%',
                 ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('% of structures'); axes[1].set_title('Grade distribution (PDB survey, schematic)')
axes[1].grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()
print('Z-score interpretation:')
for label, rng in [('Well-folded','< −3'),('Acceptable','−3 to −1'),('Poor','−1 to 0'),('Likely misfolded','> 0')]:
    print(f'  {rng:>12}  →  {label}')